In [ ]:
"""
Beach wave simulation with incidence angle θ
===========================================
Model: 2D wave equation implemented via a pseudodifferential operator in psipy.

Geometry:
  x ∈ [-Lx/2, Lx/2]  — cross-shore direction (-Lx/2 = offshore, Lx/2 = beach)
  y ∈ [-Ly/2, Ly/2]  — longshore direction (periodic)

Incidence angle θ:
  θ = 0   → waves perpendicular to the beach (normal case)
  θ > 0   → oblique waves, longshore component ky = k·sin(θ) ≠ 0
  Reflection preserves ky but inverts kx → asymmetry → oblique lateral wave
"""

from solver import *
import sympy as sp
import numpy as np
from IPython.display import HTML


In [ ]:
# ── 2. Grid Setup ──────────────────────────────────────────────────────────
Lx, Ly = 10.0, 10.0  
# Nx, Ny = 32, 32 
# Nx, Ny = 64, 64    
# Nx, Ny = 128, 128    
Nx, Ny = 256, 256    
Lt, Nt = 20.0, 800
n_frames = 400

# Create meshgrid with explicit indexing format to match gradient axes
xx, yy = np.meshgrid(np.linspace(-Lx/2, Lx/2, Nx),
                     np.linspace(-Ly/2, Ly/2, Ny), indexing='ij')



In [ ]:
# ── 1. Physical and simulation parameters ──────────────────────────────────

# ── Physical constants ──
G         = 9.81       # gravitational acceleration (m/s^2)

# ── Bathymetry ──
ALPHA     = 0.02       # beach slope: h(x) = ALPHA * (x + X0)
X0        = 9.0        # depth offset (shifts domain so h >= 0 everywhere)

# ── Wave properties ──
LAMBDA    = 4.0        # incident wavelength (m)
SPEED_FACTOR = 0.5     # velocity scaling (1.0 = full WKB, <1 = slower)

# ── Blade geometry ──
ALPHA_DEG = 6.0        # angle between the two blades (degrees)
ALPHA_R   = np.radians(ALPHA_DEG)
X1        = (Lx - 1) / 2.0   # blade 1 starts at x = +Lx/2 - 0.5 (right)
X2        = -(Lx - 1) / 2.0  # blade 2 starts at x = -Lx/2 + 0.5 (left)
BLADE_WIDTH_RATIO = 6.0       # sigma = LAMBDA / BLADE_WIDTH_RATIO
HEAVISIDE_SHARPNESS = 10.0    # k_h = HEAVISIDE_SHARPNESS / sigma
VELOCITY_SCALE_BLADE2 = 0.5   # scale factor for blade 2 velocity

# ── Wave speed model ──
C_SQUARED = 1.0        # constant: c^2 (simplified model)
# C_SQUARED = G * ALPHA * (x + X0)  # variable: full physical model

# ── Anisotropy matrix ──
# Controls directional wave propagation: A = [[a_xx, a_xy], [a_xy, a_yy]]
# a_xx > a_yy : faster cross-shore propagation
# a_xy ≠ 0    : mode coupling (waves don't propagate purely along x or y)
A_XX = 1.0        # cross-shore directional weight
A_YY = 1.0        # longshore directional weight  (< 1 → slower in y)
A_XY = 0.0        # coupling term (0 = no coupling)

BLADE1_ANGLE = 0.0       # angle of blade 1
BLADE2_ANGLE = ALPHA_R   # angle of blade 2

# ── Dissipation and Coriolis ──
GAMMA = 0.1        # bottom friction coefficient (s⁻¹)
                    # GAMMA = 0    → no dissipation
                    # GAMMA = 0.05 → moderate decay over ~20 seconds
F_CORIOLIS = -0.1   # Coriolis parameter (s⁻¹), f = 2Ω·sin(latitude)
                    # F_CORIOLIS = 0   → no Coriolis
                    # F_CORIOLIS = 0.01 → weak deflection

In [ ]:
# ── 3. SymPy symbols ────────────────────────────────────────────────────────

x, y, t       = sp.symbols('x y t',         real=True)
xi, eta       = sp.symbols('xi eta',        real=True)
g, alpha, x0  = sp.symbols('g alpha x_0',   positive=True)
f_cor         = sp.symbols('f_cor',         real=True)
u_func = Function('u') 
u = u_func(t, x, y)

# Squared phase speed
c2 = C_SQUARED

# Linear symbol with anisotropy + Coriolis
# a(x, ξ) = c²(x)·(a_xx·ξ² + 2·a_xy·ξη + a_yy·η²) + f·ξη
symbol_wave = c2 * (A_XX * xi**2 + 2*A_XY * xi*eta + A_YY * eta**2)
symbol_coriolis = F_CORIOLIS * xi * eta
aniso_symbol = symbol_wave + symbol_coriolis

print("Symbol components:")
print(f"  Wave part:     c² · (a_xx·ξ² + 2·a_xy·ξη + a_yy·η²)")
print(f"  Coriolis part: f · ξη")
print("symbol =", aniso_symbol)

In [ ]:
# ── 4. Wave equation with dissipation ────────────────────────────────────
#
# ∂²u/∂t² = -psiOp(a(x,ξ), u) - GAMMA·∂u/∂t
#
# The Coriolis term f·ξη is already in the symbol a(x,ξ).
# Only friction is a separate lower-order term.

gamma = sp.Symbol('gamma', positive=True)

equation = sp.Eq(
    diff(u, t, 2),
    -psiOp(aniso_symbol, u)
    - gamma * diff(u, t)
)

equation_num = equation.subs({g: G, alpha: ALPHA, x0: X0, gamma: GAMMA})

print("Equation:")
print(f"  ∂²u/∂t² = -psiOp(symbol, u) - {GAMMA}·∂u/∂t")

In [ ]:
# ── 5. Initial conditions ────────────────────────────────────────────────────

def _blade(xx, yy, x_center, sign, angle=0.0):
    """
    Unified blade function with variable tilt angle.
    
    Parameters:
    -----------
    xx, yy : array
        Position grids
    x_center : float
        Center position along propagation axis
    sign : int
        +1 or -1 (propagation direction, controls Heaviside orientation)
    angle : float, optional
        Tilt angle from x-axis (radians).
        angle=0     → blade parallel to y-axis
        angle≠0     → blade tilted by this angle
    """
    # Rotated coordinate perpendicular to blade front
    x_perp     = xx * np.cos(angle) + yy * np.sin(angle)
    # Reference point: blade center at y=0
    x_center_p = x_center * np.cos(angle)
    
    sigma     = LAMBDA / BLADE_WIDTH_RATIO
    gauss     = np.exp(-((x_perp - x_center_p)**2) / (2.0 * sigma**2))
    k_h       = HEAVISIDE_SHARPNESS / sigma
    heaviside = 1.0 / (1.0 + np.exp(-k_h * sign * (x_center_p - x_perp)))
    return gauss * heaviside

def initial_condition_b(xx, yy):
    blade1 = _blade(xx, yy, X1, sign=-1, angle=0.0)      # straight
    blade2 = _blade(xx, yy, X2, sign=+1, angle=ALPHA_R)  # angled
    return blade1 + blade2

def initial_velocity_b(xx, yy):
    h_local = G * ALPHA * (xx + X0)
    h_local = np.maximum(h_local, 0.05)
    c_local = np.sqrt(h_local)

    if np.allclose(xx[:, 0], xx[0, 0]):
        axis_x = 1
        axis_y = 0
        dx = xx[0, 1] - xx[0, 0]
        dy = yy[1, 0] - yy[0, 0]
    else:
        axis_x = 0
        axis_y = 1
        dx = xx[1, 0] - xx[0, 0]
        dy = yy[0, 1] - yy[0, 0]

    result_blade1 = np.zeros_like(xx)
    result_blade2 = np.zeros_like(xx)

    b1 = _blade(xx, yy, X1, sign=-1, angle=BLADE1_ANGLE)
    db1_dx = np.gradient(b1, dx, axis=axis_x)
    result_blade1 = c_local * db1_dx * SPEED_FACTOR

    b2 = _blade(xx, yy, X2, sign=+1, angle=BLADE2_ANGLE)
    db2_dx = np.gradient(b2, dx, axis=axis_x)
    db2_dy = np.gradient(b2, dy, axis=axis_y)
    db2_dir = np.cos(ALPHA_R) * db2_dx + np.sin(ALPHA_R) * db2_dy
    result_blade2 = c_local * db2_dir * SPEED_FACTOR

    # Apply the same scaling factor to blade2's velocity
    scale2 = 1.5   # adjust based on max(b1)/max(b2)
    return result_blade1 - scale2 * result_blade2

In [ ]:
# ── 5b. Plot initial conditions ──────────────────────────────────────────────
# Use the same grid as the solver (already defined in cell 2)
# xx, yy are already available with indexing='ij'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ic = initial_condition_b(xx, yy)
im = axes[0].imshow(ic.T, origin='lower', 
                    extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                    cmap='RdBu_r', aspect='auto')
axes[0].set_title('Initial elevation  η(x,y,0)')
axes[0].set_xlabel('x (cross-shore, m)')
axes[0].set_ylabel('y (longshore, m)')
plt.colorbar(im, ax=axes[0], label='η (m)')

iv = initial_velocity_b(xx, yy)
im2 = axes[1].imshow(iv.T, origin='lower',
                     extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                     cmap='RdBu_r', aspect='auto')
axes[1].set_title('Initial velocity  ∂η/∂t(x,y,0)')
axes[1].set_xlabel('x (cross-shore, m)')
axes[1].set_ylabel('y (longshore, m)')
plt.colorbar(im2, ax=axes[1], label='∂η/∂t (m/s)')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Solver instantiation and configuration ────────────────────────────────

solver = PDESolver(equation_num)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',   # periodic in both directions
    initial_condition=initial_condition_b,
    initial_velocity=initial_velocity_b,
    n_frames=n_frames,
    plot=True,                        # displays the symbol and dispersion relation
)



In [ ]:
# ── 7. Solution ────────────────────────────────────────────────────────────

frames = solver.solve()



In [ ]:
# ── 8. Visualization ─────────────────────────────────────────────────────────
plt.rcParams['animation.embed_limit'] = 2**128
# Animation: real component, wave front detection
ani = solver.animate(
    component='real',
    overlay=None,    # marks wave fronts (gradient maxima)
    mode='imshow',      # 'imshow' faster and more readable than a 3D surface
)

# Display in a Jupyter notebook:
HTML(ani.to_jshtml())